In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import evaluate
from datasets import load_dataset
from PIL import Image
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
    pipeline, 
    AutoModelForCausalLM, 
    AutoTokenizer
)

# Environment Setup & Hardware Accelerator Detection
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active Accelerator Device: {device}")

In [ ]:
# Load Dataset from 'Objects' folder
# Expected directory layout:
# Objects/
#   ├── train/
#   │   ├── class_a/
#   │   └── class_b/
#   └── validation/ (or test/)
dataset_dir = "Objects"

# Load image dataset using Hugging Face's imagefolder builder
raw_dataset = load_dataset("imagefolder", data_dir=dataset_dir)
print(f"Dataset structure:\n{raw_dataset}")

# Extract class label mappings
labels = raw_dataset["train"].features["label"].names
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

print(f"\nDiscovered {num_labels} object classes: {labels}")

# Preprocessing Transforms via AutoImageProcessor
model_checkpoint = "google/vit-base-patch16-224"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint)

In [ ]:
def transform_batch(example_batch):
    """
    Applies image resizing, normalization, and tensor conversion
    matching the original ViT pretraining pipeline.
    """
    # Ensure images are in RGB mode
    images = [img.convert("RGB") for img in example_batch["image"]]
    inputs = image_processor(images, return_tensors="pt")
    inputs["label"] = example_batch["label"]
    return inputs

In [ ]:
# Apply transformations dynamically on the fly
transformed_dataset = raw_dataset.with_transform(transform_batch)

# Model Instantiation &amp; Evaluation Metric Definition
model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True # Re-initializes classification head for custom classes
)

accuracy_metric = evaluate.load("accuracy")

In [ ]:
def compute_metrics(eval_pred):
    """
    Computes top-1 accuracy on evaluation predictions.
    Converts raw unnormalized logits to discrete class predictions via argmax.
    """
    logits, labels_gt = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels_gt)

In [ ]:
# Training Arguments &amp; Trainer Orchestration
training_args = TrainingArguments(
    output_dir="./vit-objects-finetuned",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",       # Evaluates at the end of each epoch
    save_strategy="epoch",       # Saves checkpoint at the end of each epoch
    logging_steps=10,
    load_best_model_at_end=True, # Reloads best checkpoint at completion
    metric_for_best_model="accuracy",
    greater_is_better=True,
    remove_unused_columns=False, # Required for image tensors
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=transformed_dataset["train"],
    eval_dataset=transformed_dataset.get("validation", transformed_dataset.get("test")),
    processing_class=image_processor, # Modern unified HF processing parameter
    data_collator=DefaultDataCollator(),
    compute_metrics=compute_metrics
)

# Train the ViT model
print("\nStarting Vision Transformer Fine-Tuning...")
trainer.train()

In [ ]:
# Save Fine-Tuned Model and Image Processor
save_directory = "./saved_vit_objects_model"
trainer.save_model(save_directory)
image_processor.save_pretrained(save_directory)
print(f"\nSuccessfully saved fine-tuned ViT model to '{save_directory}'.")
print(f"\nRUN THE VISUALIZATION SCRIPT AT THIS POINT!!!")
# RUN THE VISUALIZATION SCRIPT AT THIS POINT!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [ ]:
# Load Fine-Tuned ViT and Generative LLM
vit_model_path = "./saved_vit_objects_model"

# Load ViT for inference
vit_processor = AutoImageProcessor.from_pretrained(vit_model_path)
vit_model = AutoModelForImageClassification.from_pretrained(vit_model_path).to(device)
vit_model.eval()

# Load Lightweight Generative LLM (e.g., Qwen2.5-0.5B-Instruct or DistilGPT2)
llm_checkpoint = "Qwen/Qwen2.5-0.5B-Instruct"

text_generator = pipeline(
    task="text-generation",
    model=llm_checkpoint,
    device=0 if device == "cuda" else -1,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

In [ ]:
# Cascaded Perception-to-Text Pipeline Function
def predict_and_describe_object(
    image_path: str,
    confidence_threshold: float = 0.60,
    max_new_tokens: int = 60
):
    """
    Executes end-to-end vision-to-text inference with safety guardrails:
    1. Loads and displays the input test image.
    2. Runs ViT inference to extract predicted class and softmax confidence score.
    3. Mandatory Guardrail 1: Aborts text generation if confidence &lt; threshold (0.60).
    4. Mandatory Guardrail 2: Deterministic LLM generation (do_sample=False, 
       repetition_penalty=1.2, no_repeat_ngram_size=2) to block hallucination loops.
    """
    # 1. Load Image
    if not os.path.exists(image_path):
        print(f"Error: Image path '{image_path}' does not exist.")
        return

    raw_image = Image.open(image_path).convert("RGB")
    
    # Display the input image
    plt.figure(figsize=(5, 5))
    plt.imshow(raw_image)
    plt.axis("off")
    plt.title("Input Test Image", fontsize=12)
    plt.show()

    # 2. Run ViT Inference
    inputs = vit_processor(images=raw_image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = vit_model(**inputs)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=-1)
        
    top_prob, top_cat_id = torch.max(probabilities, dim=-1)
    confidence_score = top_prob.item()
    predicted_label = vit_model.config.id2label[top_cat_id.item()]

    print("=" * 60)
    print(f"📸 ViT Prediction:  {predicted_label.upper()}")
    print(f"📊 Confidence Score: {confidence_score * 100:.2f}% (Threshold: {confidence_threshold * 100:.0f}%)")
    print("=" * 60)

    # 3. Guardrail 1: Confidence Threshold Check
    if confidence_score &lt; confidence_threshold:
        print("⚠️ [GUARDRAIL TRIGGERED]: Prediction confidence is below 60%.")
        print("   Aborting LLM text generation to prevent hallucinated descriptions of uncertain visual inputs.")
        print("=" * 60)
        return {
            "predicted_label": predicted_label,
            "confidence": confidence_score,
            "description": None,
            "status": "ABORTED_LOW_CONFIDENCE"
        }

    # 4. Guardrail 2: Deterministic Anti-Hallucination Prompting &amp; Decoding Controls
    # Clean structural prompt template
    prompt = f"Description of a {predicted_label}: The {predicted_label} is typically"

    # Deterministic generation parameters to guarantee stability
    generation_output = text_generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,              # Disables random sampling (greedy decoding)
        repetition_penalty=1.2,       # Penalizes repeated tokens
        no_repeat_ngram_size=2,       # Prevents 2-gram phrase looping
        pad_token_id=text_generator.tokenizer.eos_token_id
    )

    generated_text = generation_output["generated_text"].strip()

    print("\n📝 Grounded LLM Generated Description:")
    print(f"{generated_text}")
    print("=" * 60)

    return {
        "predicted_label": predicted_label,
        "confidence": confidence_score,
        "description": generated_text,
        "status": "SUCCESS"
    }

In [ ]:
# Test on sample images
sample_test_images = [
    "Objects/test/sample_car.jpg",
    "Objects/test/sample_dog.jpg",
    "Objects/test/ambiguous_blurry_object.jpg"
]

for img_path in sample_test_images:
    if os.path.exists(img_path):
        predict_and_describe_object(img_path, confidence_threshold=0.60)